# Test VTE Status Notebook

This notebook systematically tests VTE (Venous Thromboembolism) status-related features extraction and retrieval functionality:

---

**Cells/Workflow Order:**

| Cell # | Purpose | Expected Results |
|--------|---------|------------------|
| 1-2 | Environment setup (imports, paths) | Python path configured correctly |
| 3 | Cleanup previous test outputs | No stale data remains |
| 4 | Start Elasticsearch container | Container running on port 9200 |
| 5 | Create credentials file | `test_elastic_credentials.py` generated |
| 6-7 | Populate dummy patient data + VTE status data | 5 patients with VTE in Elasticsearch cluster |
| 8 | Index refresh verification | All indices have documents |
| 9-10 | Initialize database and logging | Database at `outputs/temp_vte_status_db.sqlite` |
| 11-12 | Create pat2vec config with vte_status mode | Config object created successfully |
| 13-14 | Run pat2vec pipeline | Pipeline processes patients without errors |
| 15-16 | Extract all features from database | Features DataFrame populated |
| 17 | Output features dataframe preview | Rows of extracted features displayed |
| 18 | VTE status mode data retrieval test | `vte_status_data` contains patient features, non-empty |
| 19-20 | Feature merge functionality | `merge_vte_status_csv()` creates CSV file with data |
| 21-22 | Database and project cleanup | All temporary files deleted |
| 23 | Final verification | All assertions pass |

---

**Test Failure Conditions:**
- Any cell raises unhandled exception
- Elasticsearch container fails to start
- Empty patient list after population
- Empty DataFrame returned from feature extraction
- VTE status data retrieval returns empty or None result
- Patient count does not match expected (5 patients)
**Merge functionality produces empty result - FATAL ERROR**
- Cleanup verification fails (residual files remain)

In [ ]:
import os
import random
import shutil
import sys

import numpy as np
import pandas as pd

random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
sys.path.insert(0, pat2vec_dir)

In [ ]:
for dir_to_remove in ["vte_status_test_project"]:
    try:
        shutil.rmtree(dir_to_remove, ignore_errors=True)
    except Exception as e:
        msg = f"Failed to clean up '{dir_to_remove}' directory: {e}. "
        "Critical error - cannot start with stale data."
        raise RuntimeError(msg) from e

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

if not es_container.start():
    msg = "Failed to start Elasticsearch container. Check if Docker is running."
    raise RuntimeError(msg)

host, username, password = es_container.get_credentials()

In [ ]:
creds_filename = "test_elastic_credentials.py"

creds_content = f"""
username = "{username}"
password = "{password}"
api_key = None
hosts = ["{host}"]
"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created '{creds_filename}' pointing to {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = os.path.abspath("test_files/elastic_schemas.json")
config_populate = config_class(
    proj_name="vte_status_test_project",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import (
    populate_elastic_with_dummy_data,
)

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print(f"Population complete. Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import (
    initialize_cogstack_client,
)

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)

for index in indices:
    try:
        if cs.elastic.indices.exists(index=index):
            count = cs.elastic.count(index=index)["count"]
            print(f"  - {index:<20}: {count} documents")
        else:
            raise RuntimeError(f"Index not created: {index}")
    except Exception as e:
        raise RuntimeError(f"Error checking index {index}: {e}")

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import generate_vte_status_data
from pat2vec.util.elasticsearch_methods import ingest_data_to_elasticsearch

vte_dfs = []
for pid in patient_ids:
    df = generate_vte_status_data(
        num_rows=3,
        entered_list=[pid],
        global_start_year=int(config_populate.global_start_year),
        global_start_month=int(config_populate.global_start_month),
        global_end_year=int(config_populate.global_end_year),
        global_end_month=int(config_populate.global_end_month),
    )
    vte_dfs.append(df)

df_vte = pd.concat(vte_dfs, ignore_index=True) if len(vte_dfs) > 1 else vte_dfs[0]
df_vte = df_vte.where(pd.notnull(df_vte), None)

ingest_data_to_elasticsearch(df_vte, "observations", es_client=cs.elastic)
cs.elastic.indices.refresh(index="observations")

print(f"Ingested {len(df_vte)} VTE observations for {len(patient_ids)} patients")

In [ ]:
PROJ_NAME = "vte_status_test_project"
DB_FILENAME = "temp_vte_status_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    msg = f"Failed to remove old database file '{DB_PATH}': {e}. "
    "Critical error - cannot start with stale data."
    raise RuntimeError(msg) from e

db_connection_string = f"sqlite:///{DB_PATH}"
print(f"Database: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"vte_status": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    all_patient_list=patient_ids,
)

In [ ]:
from pat2vec.main_pat2vec import main

try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except Exception as e:
    msg = f"Failed to initialize pipeline: {e}."
    raise RuntimeError(msg) from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    msg = "No patients in patient list after initialization."
    raise RuntimeError(msg)

print(f"Processing patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = f"Failed to process patient 0 with pat_maker: {e}."
    raise RuntimeError(msg) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    msg = "FATAL ERROR: get_all_features returned an empty DataFrame."
    raise RuntimeError(msg)

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
all_features_alt = pat2vec_obj.get_all_features()

if all_features_alt.empty:
    msg = "FATAL ERROR: pat2vec_obj.get_all_features() returned an empty DataFrame."
    raise RuntimeError(msg)

print(f"pat2vec_obj.get_all_features(): {all_features_alt.shape[0]} rows retrieved.")

In [ ]:
print()
print("=== OUTPUT FEATURES DATAFRAME ===")
print(f"Shape: {all_features.shape}")
print(f"Total features: {len(all_features.columns)}")

if not all_features.empty:
    print()
    print("First 3 rows:")
    display(all_features.head(3))
else:
    msg = "DataFrame is empty after feature extraction."
    raise RuntimeError(msg)

In [ ]:
print("\n=== DEMONSTRATING DATA RETRIEVAL FOR VTE STATUS MODE ===")

all_pat_list = pat2vec_obj.all_patient_list

from pat2vec.pat2vec_get_methods.get_method_vte_status import get_vte_status_features

pat_batch = pd.DataFrame()

vte_status_data = get_vte_status_features(
    current_pat_client_id_code=all_pat_list[0],
    target_date_range=(2020, 1, 1, 2023, 12, 31),
    pat_batch=pat_batch,
    config_obj=config_obj,
)

if vte_status_data is None or (
    isinstance(vte_status_data, list) and len(vte_status_data) == 0
):
    msg = "FATAL ERROR: get_vte_status_features returned empty result."
    raise RuntimeError(msg)

if isinstance(vte_status_data, list) and len(vte_status_data) > 0:
    patient_count = len(vte_status_data)
else:
    patient_count = 1

print(f"Retrieved VTE status data for {patient_count} patient(s)")
if isinstance(vte_status_data, list):
    print(f"\nVTE columns: {list(vte_status_data[0].columns)}")
    display(vte_status_data[0])
else:
    print(f"\nVTE columns: {list(vte_status_data.columns)}")
    display(vte_status_data)

In [ ]:
# === VECTOR VALIDATION ===
feature_cols = [c for c in all_features.columns if c.startswith("vte_")]

assert len(feature_cols) > 0, "No feature columns found. Available columns: " + str(
    list(all_features.columns)
)

non_null_counts = all_features[feature_cols].notna().sum()
totally_empty = non_null_counts[non_null_counts == 0]

assert len(totally_empty) == 0, (
    f"The following feature columns are entirely null:\n"
    f"{list(totally_empty.index)}\n"
    "Vectorisation is silently failing — check the get method return value."
)

print("Feature columns (" + str(len(feature_cols)) + "): " + str(feature_cols))
print("Non-null counts per feature column:")
for col in sorted(feature_cols):
    val = all_features[col].notna().sum()
    print("  " + str(col) + ": " + str(val) + " non-null values")

In [ ]:
import pandas as pd

from pat2vec.util.post_processing_build_methods import merge_vte_status_csv

print("\n=== DEMONSTRATING FEATURE MERGE FUNCTIONALITY ===")
merged_path = merge_vte_status_csv(all_pat_list, config_obj, overwrite=True)
merged_data = pd.read_csv(merged_path)
print(f"Merged VTE status data saved to: {merged_path}")
print(f"Shape: {merged_data.shape}")

if merged_data.empty:
    msg = "FATAL ERROR: Merged VTE status DataFrame is empty."
    raise RuntimeError(msg)

print(f"\nColumns: {list(merged_data.columns)}")
display(merged_data.head())

In [ ]:
print("\n=== DATABASE AND PROJECT CLEANUP ===")

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    msg = f"Failed to remove database file '{DB_PATH}': {e}."
    raise RuntimeError(msg) from e

try:
    if os.path.exists(PROJ_NAME):
        shutil.rmtree(PROJ_NAME, ignore_errors=False)
        print(f"Removed project directory: {PROJ_NAME}")
except Exception as e:
    msg = f"Failed to remove '{PROJ_NAME}' directory: {e}."
    raise RuntimeError(msg) from e

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed Elasticsearch credentials: {creds_filename}")
except Exception as e:
    msg = f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}."
    raise RuntimeError(msg) from e

In [ ]:
print("\n=== FINAL VERIFICATION ===")

assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(PROJ_NAME), "Project directory still exists!"
assert not os.path.exists(
    creds_filename
), "Elasticsearch credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")